[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cuny-sps-datascience/citibike-data-pipeline/blob/main/teams/team-6/mvp/play_model2.ipynb)

# Play with Model 2 (calendar + forecast weather)

Model 1 knows only the calendar. Model 2 adds the weather forecast for the hour being predicted. Both were trained on
the same rows, so every comparison here is apples to apples.

Run cells top to bottom (Shift+Enter). Change the values in the **CHANGE ME** lines and re-run that cell.

- **1** load everything
- **2** pick a station + week → actual vs model 1 vs model 2, with the weather underneath
- **3** where weather helps: by hour of day, and rainy vs dry hours
- **4** the days model 1 missed worst — did model 2 fix them?
- **5** what-if: same hour, different weather
- **6** experiment: turn weather features off, retrain, compare — scored on **validation** (Jan–Feb 2026)

**Run it:** click the Colab badge (nothing to install), or locally: `pip install -r requirements.txt` then `jupyter lab` from this folder.

*Before PR #72 is merged, use this Colab link instead:* [open from the PR branch](https://colab.research.google.com/github/jamilton08/citibike-data-pipeline/blob/jonathan/mvp-6-stations-model1/teams/team-6/mvp/play_model2.ipynb)

In [ ]:
# 0. Setup — only does something in Colab / when the data isn't next to the notebook
import os, shutil, subprocess, sys
if not os.path.exists('model2/predictions.parquet'):
    os.chdir('/content')
    target = '/content/cbp'
    sources = [('https://github.com/cuny-sps-datascience/citibike-data-pipeline.git', 'main'),
               ('https://github.com/jamilton08/citibike-data-pipeline.git', 'jonathan/mvp-6-stations-model1')]
    for repo, branch in sources:
        shutil.rmtree(target, ignore_errors=True)          # clear leftovers from earlier runs
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', branch, repo, target], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f'{target}/teams/team-6/mvp/model2/predictions.parquet'):
            print(f'using {branch} from {repo}')
            break
        print(f'Model 2 files not on {branch} yet, trying next source')
    else:
        raise RuntimeError('Could not find the model 2 files on main or on the PR branch')
    os.chdir(f'{target}/teams/team-6/mvp')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pyarrow'], check=True)
print('working dir:', os.getcwd())

In [ ]:
# 1. Load predictions and both models — same rows and features as scripts/train_compare.py
import warnings; warnings.filterwarnings("ignore")
import sys, pandas as pd, numpy as np, lightgbm as lgb, matplotlib.pyplot as plt
sys.path.insert(0, 'scripts')
from mvp_data import ZONE, HOLIDAYS, FEATURES, WEATHER_FEATURES, TARGET, build_common_frame
plt.rcParams['figure.figsize'] = (12, 4)
mae = lambda y, p: float(np.abs(np.asarray(p, float) - np.asarray(y, float)).mean())

pred = pd.read_parquet('model2/predictions.parquet')
pred['target_local'] = pd.to_datetime(pred.target_hour_utc, utc=True).dt.tz_convert(ZONE)
pred['departures'] = pred.departures.astype(float)
m1 = lgb.Booster(model_file='model2/lgbm_calendar_common.txt')       # calendar only, common window
m2 = lgb.Booster(model_file='model2/lgbm_calendar_weather.txt')      # calendar + weather

STATIONS = dict(pred.drop_duplicates('station_id')[['station_name', 'station_id']].astype(str).values)
print('stations:'); [print('  ', n) for n in STATIONS]
for s in ('validation', 'test'):
    d = pred[pred.split == s]
    print(f"{s:10s} {len(d):6,d} rows | MAE baseline {mae(d.departures, d.pred_baseline):.2f} | model 1 {mae(d.departures, d.pred_model1):.2f} | model 2 {mae(d.departures, d.pred_model2):.2f}")

In [ ]:
# 2. One station, one week: actual vs model 1 vs model 2, with the weather forecast underneath
STATION = 'W 21 St & 6 Ave'      # CHANGE ME (any name printed above)
WEEK_START = '2026-07-06'        # CHANGE ME (a Monday between 2026-01-05 and 2026-08-03). 2026-07-06 was a rainy Monday.

w = pred[(pred.station_name == STATION) & (pred.horizon == 1)]
w = w[(w.target_local >= pd.Timestamp(WEEK_START, tz=ZONE)) & (w.target_local < pd.Timestamp(WEEK_START, tz=ZONE) + pd.Timedelta(days=7))]
fig, (a, b) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
a.bar(w.target_local, w.departures, width=1/24, color='lightgray', label='actual')
a.plot(w.target_local, w.pred_model1, lw=1.5, ls='--', label='model 1 (calendar)')
a.plot(w.target_local, w.pred_model2, lw=2, label='model 2 (+ weather)')
a.set_title(f'{STATION} — week of {WEEK_START}'); a.set_ylabel('departures / hour'); a.legend()
b.bar(w.target_local, w.forecast_precipitation_in, width=1/24, label='forecast precipitation (in)')
b.set_ylabel('precip (in)'); t = b.twinx(); t.plot(w.target_local, w.forecast_temperature_f, color='gray', lw=1); t.set_ylabel('temp (°F)')
plt.tight_layout(); plt.show()
print(f'MAE this week — model 1 {mae(w.departures, w.pred_model1):.2f} | model 2 {mae(w.departures, w.pred_model2):.2f}')

In [ ]:
# 3. Where does weather help? (test period, Mar 1 – Aug 10 2026, all stations)
t = pred[(pred.split == 'test') & (pred.horizon == 1)].copy()
t['hour'] = t.target_local.dt.hour
t['err1'] = (t.pred_model1 - t.departures).abs(); t['err2'] = (t.pred_model2 - t.departures).abs()
t['weather'] = np.where(t.forecast_precipitation_in > 0.005, 'rainy hours', 'dry hours')

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={'width_ratios': [3, 1]})
t.groupby('hour')[['err1', 'err2']].mean().rename(columns={'err1': 'model 1', 'err2': 'model 2'}).plot(kind='bar', width=0.8, ax=a)
a.set_title('Average miss by hour of day'); a.set_ylabel('MAE (bikes)'); a.set_xlabel('hour')
g = t.groupby('weather')[['err1', 'err2']].mean().rename(columns={'err1': 'model 1', 'err2': 'model 2'})
g.plot(kind='bar', width=0.7, ax=b, legend=False); b.set_title('Rainy vs dry hours'); b.set_xlabel(''); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
print(g.round(2).assign(hours=t.groupby('weather').size()))

In [ ]:
# 4. The days model 1 missed worst — did model 2 fix them? (test period)
d = t.assign(date=t.target_local.dt.date).groupby(['date', 'station_name']).agg(
        actual=('departures', 'sum'), model1=('pred_model1', 'sum'), model2=('pred_model2', 'sum'),
        precip_in=('forecast_precipitation_in', 'sum'), temp_f=('forecast_temperature_f', 'mean')).reset_index()
d['miss1'] = d.model1 - d.actual; d['miss2'] = d.model2 - d.actual
print('Days model 1 OVER-predicted most:')
display(d.sort_values('miss1', ascending=False).head(10).round(1))
print('Days model 1 UNDER-predicted most:')
display(d.sort_values('miss1').head(10).round(1))
print('Rainy days (precip_in > 0): model 2 should cut the miss. Days with no rain that both models miss')
print('(for example a station taken out of service) are not something weather can fix.')

In [ ]:
# 5. What-if: same station and hour, different weather
STATION = 'Broadway & E 14 St'   # CHANGE ME
WHEN = '2026-07-15 17:00'        # CHANGE ME (local NYC time)
TEMP_F, WIND_MPH = 80, 6         # CHANGE ME

def row_for(station_name, when_local, temp_f, precip_in=0.0, snow_in=0.0, wind_mph=6.0, horizon=1):
    ts = pd.Timestamp(when_local, tz=ZONE) + pd.Timedelta(hours=horizon - 1)
    r = pd.DataFrame([{'station_id': STATIONS[station_name], 'horizon': horizon, 't_hour': ts.hour, 't_dow': ts.dayofweek,
                       't_month': ts.month, 't_doy': ts.dayofyear, 't_weekend': int(ts.dayofweek >= 5),
                       't_holiday': int(ts.date() in HOLIDAYS), 'forecast_temperature_f': temp_f,
                       'forecast_precipitation_in': precip_in, 'forecast_rain_in': precip_in if snow_in == 0 else 0.0,
                       'forecast_snowfall_in': snow_in, 'forecast_wind_speed_mph': wind_mph}])
    r['station_id'] = pd.Categorical(r.station_id, categories=sorted(STATIONS.values()))
    return r

print(f'{STATION}, {pd.Timestamp(WHEN):%a %b %d %Y %I %p}')
print(f'  model 1 (calendar only):           {m1.predict(row_for(STATION, WHEN, TEMP_F)[FEATURES])[0]:5.1f} departures')
for label, kw in [('dry', {}), ('light rain (0.05 in)', {'precip_in': 0.05}), ('heavy rain (0.30 in)', {'precip_in': 0.30})]:
    X = row_for(STATION, WHEN, TEMP_F, wind_mph=WIND_MPH, **kw)[FEATURES + WEATHER_FEATURES]
    print(f'  model 2, {TEMP_F}°F, {label:22s} {m2.predict(X)[0]:5.1f} departures')

temps = np.arange(10, 101, 5)
for label, p in [('dry', 0.0), ('rain 0.10 in', 0.10)]:
    y = [m2.predict(row_for(STATION, WHEN, tf, precip_in=p, wind_mph=WIND_MPH)[FEATURES + WEATHER_FEATURES])[0] for tf in temps]
    plt.plot(temps, y, marker='o', label=label)
plt.title(f'Model 2: predicted departures vs forecast temperature — {STATION}, {pd.Timestamp(WHEN):%a %I %p}')
plt.xlabel('forecast temperature (°F)'); plt.ylabel('predicted departures / hour'); plt.legend(); plt.show()
print('Temperatures far outside what this date normally sees are a guess: the model has few examples of them.')

In [ ]:
# 6. Experiment: turn weather features off, retrain, compare VALIDATION MAE (~15 s per run)
#    Validation (Jan–Feb 2026) is for choices like this. The test period is only reported by scripts/train_compare.py.
full = build_common_frame()
tr, va = full[full.split == 'train'], full[full.split == 'validation']

def run(features):
    m = lgb.LGBMRegressor(objective='poisson', n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.9, random_state=622, verbose=-1)
    m.fit(tr[features], tr[TARGET], eval_set=[(va[features], va[TARGET])], eval_metric='poisson',
          categorical_feature=['station_id'], callbacks=[lgb.early_stopping(150, verbose=False)])
    return mae(va[TARGET], m.predict(va[features]))

# CHANGE ME: add/remove experiments
experiments = {
    'calendar only (model 1)':        FEATURES,
    'calendar + all weather (model 2)': FEATURES + WEATHER_FEATURES,
    'weather without temperature':    FEATURES + [f for f in WEATHER_FEATURES if f != 'forecast_temperature_f'],
    'weather without rain/snow':      FEATURES + ['forecast_temperature_f', 'forecast_wind_speed_mph'],
    'temperature only':               FEATURES + ['forecast_temperature_f'],
}
for name, feats in experiments.items():
    print(f'{name:34s} validation MAE {run(feats):.3f}')